In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.primitives import StatevectorSampler
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
import time
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer import AerSimulator
from qiskit.quantum_info import partial_trace
from iqm.qiskit_iqm.move_gate import MOVE_GATE_UNITARY
from sklearn.datasets import load_digits
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

SHOTS = 8192

# Noiseless simulator that returns shots
ideal_sampler = StatevectorSampler(seed=42)

# Aer device for the noisy simulations: GPU if available (needs the qiskit-aer-gpu package), otherwise CPU
USE_GPU = True
AER_DEVICES = AerSimulator().available_devices()
DEVICE = "GPU" if USE_GPU and "GPU" in AER_DEVICES else "CPU"
print("Aer devices:", AER_DEVICES, "-> using", DEVICE)

1. VLQ calibration data

In [ ]:
# Placeholder values, overwritten in section 1b (except the gate durations)
N_QB = 24

PRX_TIME_NS  = 40                         # single-qubit gate (r) duration
CZ_TIME_NS   = 60                         # CZ (qubit-resonator) duration
MOVE_TIME_NS = 100                        # MOVE (qubit <-> resonator) duration

T1_us     = np.full(N_QB, 40.0)           # energy relaxation time T1 (µs)
T2_us     = np.full(N_QB, 30.0)           # dephasing time T2 (µs)
err_1q    = np.full(N_QB, 1e-3)           # single-qubit gate error (1 - fidelity)
p1_given0 = np.full(N_QB, 0.02)           # readout error P(1|0)
p0_given1 = np.full(N_QB, 0.04)           # readout error P(0|1)
err_cz    = np.full(N_QB, 1e-2)           # CZ error between QB(i+1) and the resonator
err_move  = np.full(N_QB, 1e-2)           # MOVE error between QB(i+1) and the resonator

T1_res_us, T2_res_us = 40.0, 30.0         # resonator coherence times (µs)

for arr in [T1_us, T2_us, err_1q, p1_given0, p0_given1, err_cz, err_move]:
    assert len(arr) == N_QB

In [ ]:
import json

with open("vlq-calset-default-metrics.json") as f:
    calibration = json.load(f)

metrics = {obs["dut_field"]: obs["value"] for obs in calibration["observations"] if not obs["invalid"]}
print("Calibration date:", calibration["created_timestamp"])

for i in range(N_QB):
    q = f"QB{i + 1}"
    T1_us[i] = metrics[f"characterization.model.{q}.t1_time"] * 1e6
    T2_us[i] = metrics.get(f"characterization.model.{q}.t2_time",
                           metrics[f"characterization.model.{q}.t2_echo_time"]) * 1e6
    err_1q[i]    = 1 - metrics[f"metrics.rb.prx.drag_crf.{q}.fidelity:par=d1"]
    p1_given0[i] = metrics[f"metrics.ssro.measure.constant.{q}.error_0_to_1"]
    p0_given1[i] = metrics[f"metrics.ssro.measure.constant.{q}.error_1_to_0"]
    err_cz[i]    = 1 - metrics[f"metrics.irb.cz.crf_crf.{q}__COMPR1.fidelity"]
    err_move[i]  = (1 - metrics[f"metrics.irb.move.crf_crf.{q}__COMPR1.fidelity:n_interleaved_gates=2"]) / 2

T1_res_us = metrics["characterization.model.COMPR1.t1_time"] * 1e6
T2_res_us = metrics["characterization.model.COMPR1.t2_time"] * 1e6

print(f"{'qubit':6s} {'T1(us)':>7s} {'T2(us)':>7s} {'err_1q':>8s} {'P(1|0)':>7s} {'P(0|1)':>7s} {'err_cz':>7s} {'err_move':>8s}")
for i in range(N_QB):
    print(f"QB{i + 1:<4d} {T1_us[i]:7.1f} {T2_us[i]:7.1f} {err_1q[i]:8.4f} {p1_given0[i]:7.3f} "
          f"{p0_given1[i]:7.3f} {err_cz[i]:7.4f} {err_move[i]:8.4f}")
print(f"COMPR1 {T1_res_us:7.1f} {T2_res_us:7.1f}")

# Qubits used in the simulation: ranked by two-qubit (CZ + 2 MOVE) and readout errors
score = err_cz + 2 * err_move + (p1_given0 + p0_given1) / 2
BEST_QUBITS = [f"QB{i + 1}" for i in np.argsort(score)]
print("\nQubits ranked from best to worst:", BEST_QUBITS)

QUBIT_SETS = {"best": BEST_QUBITS, "worst": BEST_QUBITS[::-1]}

Calibration of VLQ per qubit (reference figure).

In [ ]:
calibration_table = {"T1 (µs)": T1_us, "T2 (µs)": T2_us, "prx error": err_1q,
                     "CZ error": err_cz, "MOVE error": err_move, "Readout error": (p1_given0 + p0_given1) / 2}
qubit_labels = [f"QB{i + 1}" for i in range(N_QB)]
colors = ["tab:blue"] * N_QB

fig, ax = plt.subplots(len(calibration_table), 1, figsize=(12, 1.8 * len(calibration_table)), sharex=True)
for a, (label, values) in zip(ax, calibration_table.items()):
    a.bar(qubit_labels, values, color=colors)
    a.set_ylabel(label)
ax[-1].tick_params(axis="x", rotation=90)
plt.tight_layout(); plt.savefig("fig_calibration.jpg", bbox_inches="tight"); plt.show()

1c. Qubits used (no selection)

In [ ]:
QUBIT_SETS["fixed"] = [f"QB{i + 1}" for i in range(N_QB)]
print("Qubits used for n = 9:", QUBIT_SETS["fixed"][:9])

2. Star-topology circuits and noise model

In [ ]:
def to_star(qc_cz, n):
    # qc_cz: circuit on n data qubits with rz, sx, x, cz (and measurements)
    # returns the same circuit on n data qubits + resonator (index n), with CZs routed through the resonator
    R = n
    star = QuantumCircuit(n + 1)
    if qc_cz.num_clbits:
        star.add_register(ClassicalRegister(qc_cz.num_clbits, "meas"))
    holder = None                                    # data qubit whose state is in the resonator

    def move_out():
        nonlocal holder
        if holder is not None:
            star.unitary(MOVE_GATE_UNITARY, [holder, R], label="move")
            holder = None

    for inst in qc_cz.data:
        name = inst.operation.name
        qubits = [qc_cz.find_bit(q).index for q in inst.qubits]
        if name == "barrier":
            continue
        if name == "cz":
            a, b = qubits
            if holder == b:
                a, b = b, a
            if holder != a:                          # bring a into the resonator
                move_out()
                star.unitary(MOVE_GATE_UNITARY, [a, R], label="move")
                holder = a
            star.cz(b, R)                            # CZ(b, resonator) = CZ(b, a)
        elif name == "measure":
            move_out()
            star.measure(qubits[0], qc_cz.find_bit(inst.clbits[0]).index)
        else:                                        # single-qubit gate
            if holder in qubits:
                move_out()
            star.append(inst.operation, qubits)
    move_out()
    return star


def calibration_index(name):
    # component name -> index in the calibration arrays (None for the resonator)
    return int(name[2:]) - 1 if name.startswith("QB") else None


ALL_SOURCES = ("thermal", "prx", "cz", "move", "readout")


def build_noise_model(names, t_scale=1.0, sources=ALL_SOURCES):
    # names[k] = component (QBx or COMPR1) of qubit k of the simulated circuit
    # sources = error sources switched on:
    #   "thermal": T1/T2 relaxation during each gate, "prx": single-qubit gate error,
    #   "cz": CZ error, "move": MOVE error, "readout": measurement error
    noise_model = NoiseModel(basis_gates=["rz", "sx", "x", "cz", "unitary"])

    def relaxation(p, time_ns):
        c = calibration_index(p)
        T1 = (T1_us[c] if c is not None else T1_res_us) * 1e3 * t_scale   # µs -> ns
        T2 = (T2_us[c] if c is not None else T2_res_us) * 1e3 * t_scale
        return thermal_relaxation_error(T1, min(T2, 2 * T1), time_ns)    # T2 <= 2 T1

    def combine(depol, relax):
        if depol is None:
            return relax
        return depol if relax is None else depol.compose(relax)

    # single-qubit gates sx, x (rz is virtual: no error) and readout
    for k, p in enumerate(names):
        c = calibration_index(p)
        if c is None:                                   # resonator: no single-qubit gates, no readout
            continue
        depol = depolarizing_error(2 * err_1q[c], 1) if "prx" in sources else None
        relax = relaxation(p, PRX_TIME_NS) if "thermal" in sources else None
        error = combine(depol, relax)
        if error is not None:
            noise_model.add_quantum_error(error, ["sx", "x"], [k])
        if "readout" in sources:
            e0, e1 = p1_given0[c], p0_given1[c]
            noise_model.add_readout_error(ReadoutError([[1 - e0, e0], [e1, 1 - e1]]), [k])

    # two-qubit gates: always between a qubit and the resonator
    for k0, p0 in enumerate(names):
        for k1, p1 in enumerate(names):
            if k0 == k1:
                continue
            c = calibration_index(p0) if calibration_index(p0) is not None else calibration_index(p1)
            for gate, time_ns, err in [("cz", CZ_TIME_NS, err_cz), ("move", MOVE_TIME_NS, err_move)]:
                depol = depolarizing_error(4 / 3 * err[c], 2) if gate in sources else None
                relax = relaxation(p0, time_ns).expand(relaxation(p1, time_ns)) if "thermal" in sources else None
                error = combine(depol, relax)
                if error is not None:
                    noise_model.add_quantum_error(error, gate, [k0, k1])
    return noise_model


VLQ_NOISE = dict(sources=ALL_SOURCES)

3. Dataset (0, 2, 7, 9)

In [ ]:
N_TRAIN, N_TEST = 250, 250                       # images per digit

from sklearn.datasets import fetch_openml
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
data, target = mnist.data / 255.0, mnist.target.astype(int)

PAN_DIGITS = [0, 2, 7, 9]

rng_pair = np.random.default_rng(seed=0)
binary_digits = sorted(rng_pair.choice(PAN_DIGITS, size=2, replace=False).tolist())

tasks = {f"{binary_digits[0]} vs {binary_digits[1]}": binary_digits,
         "0, 2, 7, 9": PAN_DIGITS}
print(tasks)


def make_split(digit_list, seed=0):
    rng_digits = np.random.default_rng(seed)
    train_idx, test_idx = [], []
    for d in digit_list:
        idx_d = np.where(target == d)[0]              # all images of digit d
        assert len(idx_d) >= N_TRAIN + N_TEST, f"only {len(idx_d)} images of digit {d}"
        idx_d = rng_digits.permutation(idx_d)         # shuffle them
        train_idx += list(idx_d[:N_TRAIN])
        test_idx  += list(idx_d[N_TRAIN:N_TRAIN + N_TEST])
    return data[train_idx], target[train_idx], data[test_idx], target[test_idx]

4. Compression to n features: truncated SVD + scaling to [0, 1]

In [ ]:
def compression(X_train_raw, X_test_raw, n):
    svd = TruncatedSVD(n_components=n, random_state=0)
    svd.fit(X_train_raw)                         # learns the n directions, using only train
    Z_train = svd.transform(X_train_raw)
    Z_test  = svd.transform(X_test_raw)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(Z_train)                          # learns min and max, using only train
    X_train = scaler.transform(Z_train)
    X_test  = np.clip(scaler.transform(Z_test), 0, 1)
    return X_train, X_test

5. Kernel circuit: U(x_j) followed by U(x_i)^dagger

In [ ]:
def feature_map(x, n):          # n = number of qubits = number of features
    qc = QuantumCircuit(n)

    # Feature map, 4 layers (Sec. 3: "increase the depth to 4")
    for _ in range(4):
        for i in range(n):
            qc.h(i)
            qc.rz(2 * x[i], i)
        for i in range(n - 1):          # RZZ on neighbouring pairs (0,1), (1,2), ...
            x_ij = 2 * (np.pi - x[i]) * (np.pi - x[i + 1])
            qc.rzz(x_ij, i, i + 1)
    return qc


def kernel_circuit(x_i, x_j, n):
    qc = feature_map(x_j, n)                                 # U(x_j)
    qc.compose(feature_map(x_i, n).inverse(), inplace=True)  # U(x_i)^dagger
    qc.measure_all()
    return qc

6. Kernel from the quantum states (one simulation per image)

In [ ]:
rng_shots = np.random.default_rng(42)


def sample_shots(P, shots=SHOTS):
    return rng_shots.binomial(shots, np.clip(P, 0, 1)) / shots


# noiseless: statevectors
def statevectors(X, n):
    return np.array([Statevector(feature_map(x, n)).data for x in X])


# noisy: density matrices on the VLQ noise model
state_templates = {}

def noisy_state_template(n, qubits):
    # parametrized feature map U(x) on the first n qubits of `qubits` + resonator, compiled once per (n, qubits)
    key = (n, tuple(qubits[:n]))
    if key not in state_templates:
        t0 = time.time()
        names = list(qubits[:n]) + ["COMPR1"]           # circuit qubit k is names[k]
        x = ParameterVector("x", n)
        qc_cz = transpile(feature_map(x, n), basis_gates=["rz", "sx", "x", "cz"],
                          optimization_level=1, seed_transpiler=42)
        qc_dm = to_star(qc_cz, n)
        qc_dm.save_density_matrix()
        state_templates[key] = (qc_dm, names, list(x), [n])
        print(f"  n = {n}: compiled in {time.time() - t0:.1f} s, components {names}, ops {dict(qc_dm.count_ops())}")
    return state_templates[key]


def noisy_states(X, n, noise, chunk=64):
    noise = dict(noise)
    qubits = QUBIT_SETS[noise.pop("qubits", "best")]       # which physical qubits are used
    qc_dm, names, x_params, resonator = noisy_state_template(n, qubits)
    present = set(qc_dm.parameters)
    simulator = AerSimulator(method="density_matrix", device=DEVICE, noise_model=build_noise_model(names, **noise))

    states = []
    for start in range(0, len(X), chunk):
        Xc = X[start:start + chunk]
        binds = {p: [float(x[m]) for x in Xc] for m, p in enumerate(x_params) if p in present}
        result = simulator.run(qc_dm, parameter_binds=[binds]).result()
        for k in range(len(Xc)):
            rho = result.data(k)["density_matrix"]
            if resonator:
                rho = partial_trace(rho, resonator)       # resonator is back in |0>
            states.append(np.asarray(rho.data, dtype=np.complex64).ravel())
    return np.array(states)


# kernel matrices
def kernel_matrices(X_train, X_test, n, noise):
    if noise is None:
        A, B = statevectors(X_train, n), statevectors(X_test, n)
        P_train, P_test = np.abs(A.conj() @ A.T) ** 2, np.abs(B.conj() @ A.T) ** 2
    else:
        A, B = noisy_states(X_train, n, noise), noisy_states(X_test, n, noise)
        P_train, P_test = np.real(A @ A.conj().T), np.real(B @ A.conj().T)   # Tr[rho rho']

    N = len(X_train)
    upper = np.triu_indices(N, k=1)
    K_train = np.eye(N)                                   # diagonal = 1, as before
    K_train[upper] = sample_shots(P_train[upper])
    K_train = np.triu(K_train) + np.triu(K_train, 1).T    # symmetric
    K_test = sample_shots(P_test)
    return K_train, K_test

In [ ]:
# Quick check: statevector kernel vs the compute-uncompute circuit on the sampler
x_a, x_b = np.array([0.2, 0.7, 0.4]), np.array([0.5, 0.1, 0.9])
S = statevectors([x_a, x_b], 3)
print("statevector :", abs(S[0].conj() @ S[1]) ** 2)
r = ideal_sampler.run([kernel_circuit(x_a, x_b, 3)], shots=100000).result()[0]
print("sampler     :", r.data.meas.get_counts().get("000", 0) / 100000)

# noisy states: trace must be 1 and purity < 1
R = noisy_states([x_a, x_b], 3, VLQ_NOISE)
print("trace       :", np.real(np.trace(R[0].reshape(8, 8))), " purity:", np.real(R[0] @ R[0].conj()))

In [ ]:
n_show = 3
feature_map(ParameterVector("x", n_show), n_show).draw("mpl")

In [ ]:
qc_dm, names, _, _ = noisy_state_template(n_show, QUBIT_SETS["best"])
print("components:", names)
print("gate counts:", dict(qc_dm.count_ops()))
qc_dm.draw("mpl", fold=40)

7. General case: n = 2, ..., 9

In [ ]:
def case2_gen(n, digit_list, noise, seed=0):
    X_train_raw, y_train, X_test_raw, y_test = make_split(digit_list, seed=seed)
    X_train, X_test = compression(X_train_raw, X_test_raw, n)

    K_train, K_test = kernel_matrices(X_train, X_test, n, noise)

    svm = SVC(kernel="precomputed")
    svm.fit(K_train, y_train)
    acc = svm.score(K_test, y_test)
    return acc, K_train


def nmse(K_sim, K):                              # Eq. (3) of Melo et al.
    return np.sum((K_sim - K) ** 2) / np.sum(K_sim ** 2)

8. Which error source matters most?

In [ ]:
configs = {"Noiseless":                 None,
           "Only T1/T2":                dict(qubits="fixed", sources=("thermal",)),
           "Only prx":                  dict(qubits="fixed", sources=("prx",)),
           "Only CZ":                   dict(qubits="fixed", sources=("cz",)),
           "Only MOVE":                 dict(qubits="fixed", sources=("move",)),
           "Depolarizing (fidelities)": dict(qubits="fixed", sources=("prx", "cz", "move")),
           "All errors":                dict(qubits="fixed", sources=ALL_SOURCES)}
n_list = [2, 3, 4, 5, 6, 7, 8, 9]

analysis = {}
for task, digit_list in tasks.items():
    K_ideal = {}
    for label, noise in configs.items():          # "Noiseless" first: its kernels are the reference for the NMSE
        t0 = time.time()
        accs, errs = [], []
        for n in n_list:
            acc, K = case2_gen(n, digit_list, noise=noise)
            if noise is None:
                K_ideal[n] = K
            accs.append(acc)
            errs.append(0.0 if noise is None else nmse(K_ideal[n], K))
        analysis[(task, label)] = {"acc": accs, "nmse": errs}
        print(f"{task:10s} | {label:27s} acc = {np.round(accs, 2)}  NMSE = {np.round(errs, 3)}  ({(time.time() - t0) / 60:.1f} min)")
        np.save(f"results_noise_analysis_{DATASET}_{N_TRAIN}_{N_TEST}.npy", analysis, allow_pickle=True)

In [ ]:
fig, ax = plt.subplots(len(tasks), 2, figsize=(13, 4 * len(tasks)))
for row, task in enumerate(tasks):
    for label in configs:
        r = analysis[(task, label)]
        ax[row, 0].plot(n_list, r["acc"], "o-", label=label)
        if label != "Noiseless":
            ax[row, 1].plot(n_list, r["nmse"], "o-", label=label)
    ax[row, 0].set_ylabel("Classification accuracy"); ax[row, 1].set_ylabel("Kernel NMSE vs noiseless")
    for a in ax[row]:
        a.set_title(task); a.set_xlabel("Number of qubits"); a.set_xticks(n_list); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig("fig_noise_sources.jpg", bbox_inches="tight"); plt.show()

9. Accuracy vs coherence time

In [ ]:
n_sweep = 5
t_scales = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]

sweep = {task: [] for task in tasks}
for s in t_scales:
    noise_s = dict(qubits="fixed", t_scale=s, sources=("thermal",))
    for task, digit_list in tasks.items():
        acc, _ = case2_gen(n_sweep, digit_list, noise=noise_s)
        sweep[task].append(acc)
        print(f"{task} | T scale = {s}: accuracy = {acc:.2f}")

for task in tasks:
    plt.semilogx(t_scales, sweep[task], "o-", label=task)
plt.axvline(1, color="gray", ls="--", label="VLQ T1, T2")
plt.xlabel("T1, T2 scale factor"); plt.ylabel("Classification accuracy")
plt.title(f"n = {n_sweep}"); plt.legend()
plt.savefig("fig_noise_T_sweep.jpg", bbox_inches="tight")
plt.show()